# Lab 07–08: Structured Streaming, Event Windows & Watermarks

**Z5008 Big Data Lab (MTech Core) · IIT Madras Zanzibar**

Event-time tumbling windows, bounded lateness watermarks, and straggler filtering.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5008/labs/BD03_Structured_Streaming_Event_Windows.ipynb)

Run cells top to bottom (`Runtime > Run all`). Spark steps include pure-Python fallbacks when executed outside a Spark/JVM container.

## 1. Event-Time Tumbling Windows & Watermark State Eviction

A watermark $W = \max(\text{event\_time}) - \text{allowed\_lateness}$ lets streaming engines emit finalized window totals and discard bounded-late stragglers without unbounded memory growth.

In [1]:
from collections import defaultdict


def process_stream(events, win_sec=10, delay=5):
    max_t = 0
    sums, dropped = defaultdict(float), []
    for t, key, amt in events:
        max_t = max(max_t, t)
        wm = max_t - delay
        w_start = (t // win_sec) * win_sec
        if w_start + win_sec <= wm:
            dropped.append((t, key, amt))
        else:
            sums[(w_start, key)] += amt
    return dict(sums), dropped


events = [
    (2, "M-Pesa", 40.0), (8, "M-Pesa", 60.0), (12, "TigoPesa", 25.0),
    (19, "M-Pesa", 50.0), (4, "M-Pesa", 99.0), (16, "TigoPesa", 15.0),
]
sums, dropped = process_stream(events)
print("Window aggregates:", sums)
print("Dropped late events:", dropped)
assert sums[(0, "M-Pesa")] == 100.0 and dropped == [(4, "M-Pesa", 99.0)]


Window aggregates: {(0, 'M-Pesa'): 100.0, (10, 'TigoPesa'): 40.0, (10, 'M-Pesa'): 50.0}
Dropped late events: [(4, 'M-Pesa', 99.0)]
